# TempoBIS on Kaggle — end-to-end runner

One notebook, one phase per section. Run a section, check its output, then continue.

**Before running**
1. *Add Input* → search **WL-BISINDO** (Kindy, Leonali & Lucky, 2025) and add it.
2. *Settings* → **Internet: On** (needed for `pip install` and the MediaPipe model files).
3. *Accelerator* → GPU (T4) is optional; it only speeds up training. Extraction runs on CPU.

Dataset license: **CC BY-NC 4.0** — fine for a non-commercial portfolio with attribution.
Do not re-upload the videos to GitHub.

## Setup — get the code

In [ ]:
import os, sys, subprocess, glob

REPO_URL = ""   # e.g. "https://github.com/<your-username>/TempoBIS.git" once the repo is public

if REPO_URL:
    if not os.path.exists("/kaggle/working/TempoBIS"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/TempoBIS"], check=True)
    CODE = "/kaggle/working/TempoBIS"
else:
    # code uploaded as a Kaggle dataset (zip of this repo)
    hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    hits = [h for h in hits if "tempobis" in open(h).read().lower()]
    assert hits, "Set REPO_URL or add the TempoBIS code as an input dataset"
    import shutil
    shutil.copytree(os.path.dirname(hits[0]), "/kaggle/working/TempoBIS", dirs_exist_ok=True)
    CODE = "/kaggle/working/TempoBIS"

os.chdir(CODE)
!pip install -q -e . 2>&1 | tail -2
sys.path.insert(0, f"{CODE}/src")   # editable installs are only picked up by new kernels
print("code at", CODE)

In [ ]:
# Check: unit tests (includes the GRU test, which needs PyTorch — available on Kaggle)
!python -m pytest -q 2>&1 | tail -3

In [ ]:
import yaml
from tempobis.config import load_config, _merge
CONFIG = "configs/default.yaml"
cfg = load_config(CONFIG, overrides={"paths": {"work_dir": "/kaggle/working/outputs"}})
with open("configs/kaggle.yaml", "w") as f:
    yaml.safe_dump(cfg, f)
quick = _merge(cfg, yaml.safe_load(open("configs/quick.yaml")))  # same landmarks, separate results folder
with open("configs/kaggle_quick.yaml", "w") as f:
    yaml.safe_dump(quick, f)
stress = _merge(cfg, yaml.safe_load(open("configs/stress.yaml")))  # harder stress test, own results folder
with open("configs/kaggle_stress.yaml", "w") as f:
    yaml.safe_dump(stress, f)
CFG, QCFG, SCFG = "configs/kaggle.yaml", "configs/kaggle_quick.yaml", "configs/kaggle_stress.yaml"
print(yaml.safe_dump(cfg["paths"]))

## Phase 0 — Dataset audit
Goal: understand what is actually in the dataset before touching any model.
Check: number of videos / glosses / signers, videos per signer, fps, frame counts, broken or duplicate files.

In [ ]:
!python -m tempobis audit --config $CFG

In [ ]:
import pandas as pd
audit = pd.read_csv("/kaggle/working/outputs/audit/audit_videos.csv")
display(pd.crosstab(audit.gloss, audit.signer).style.background_gradient(axis=None))
audit[["decoded_frames", "duration_s", "fps"]].describe()

**Write down** (for the README / report): totals, videos per signer, fps, frame-count range, anything broken.
If a signer has far fewer videos, note it — it affects the leave-one-signer-out folds.

## Phase 1 — MediaPipe pilot (10 videos)
Goal: landmarks can be extracted and look right. Check: detection rates and one visual overlay.

In [ ]:
!python -m tempobis pilot --config $CFG --n 10

In [ ]:
import numpy as np, matplotlib.pyplot as plt, cv2
from tempobis.landmarks import load_landmarks
from tempobis.features import frame_features, wrist_heights
from tempobis.temporal import motion_energy, active_segment, trim_bounds

pilot = sorted(glob.glob("/kaggle/working/outputs/pilot/*.npz"))
vid = os.path.basename(pilot[0])[:-4]
raw = load_landmarks(pilot[0])
row = audit[audit.video_id == vid].iloc[0]
ts = np.linspace(0, len(raw["pose"]) - 1, 4).astype(int)
# Read frames sequentially: seeking with CAP_PROP_POS_FRAMES is unreliable for these videos.
cap = cv2.VideoCapture(row.path); frames, i = {}, 0
while len(frames) < len(set(ts)):
    ok, fr = cap.read()
    if not ok:
        break
    if i in ts:
        frames[i] = fr
    i += 1
cap.release()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, t in zip(axes, ts):
    fr = frames.get(t)
    if fr is None:
        ax.set_title(f"frame {t} unreadable"); ax.axis("off"); continue
    h, w = fr.shape[:2]; ax.imshow(fr[..., ::-1])
    for key, c in (("pose", "y"), ("lhand", "c"), ("rhand", "m")):
        p = raw[key][t]; ax.scatter(p[:, 0] * w, p[:, 1] * h, s=6, c=c)
    ax.set_title(f"{vid} frame {t}"); ax.axis("off")
plt.show()

x = frame_features(raw); e = motion_energy(x); a, b = active_segment(x); lo, hi = trim_bounds(x)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(e, label="motion energy"); ax.axvspan(a, b, alpha=0.15, label="active (wrist height)")
ax.axvline(lo, ls="--", c="k"); ax.axvline(hi, ls="--", c="k", label="trim bounds")
ax.legend(); ax.set_xlabel("frame"); plt.show()

**Check before continuing:** hand points sit on the hands; left/right are not swapped;
the active segment and trim bounds cover the sign. If hands are often missed, try `pose_model: full`
or lower `min_detection_confidence` in the config before extracting everything.

## Phase 2 — Extract the whole dataset (resumable; ~30–60 min on 4 CPU workers)

In [ ]:
!python -m tempobis extract --config $CFG

In [ ]:
st = pd.read_csv("/kaggle/working/outputs/landmark_stats.csv")
print(len(st), "videos extracted")
st[["frames", "pose_rate", "lhand_rate", "rhand_rate", "any_hand_rate"]].describe()

**Check:** every video extracted; note videos with very low `any_hand_rate` (used later in error analysis).

## Phase 3 — Quick baseline sanity run
Official signer-independent split (test signer 4), one seed, fewer epochs. Goal: the model learns
something sensible without leakage before spending GPU time on the full experiment.

In [ ]:
!python -m tempobis train --config $QCFG 2>&1 | tail -30

Results go to `outputs/results_quick/`. If clean macro-F1 is near chance (1/32 ≈ 0.03),
stop and debug (labels, features) before Phase 4–5.

## Phases 4–5 — Full experiment: baseline vs motion-aware under the speed stress test
5 leave-one-signer-out folds × 3 seeds × 4 methods (uniform, trim+uniform, motion-aware,
uniform+speed-aug), each evaluated at 5 speed factors × 2 perturbation types. Resumable: rerun the
cell if the session restarts (copy `outputs/results` to a dataset to keep it between sessions).

In [ ]:
!python -m tempobis train --config $CFG 2>&1 | tail -80

In [ ]:
from IPython.display import Markdown, Image
display(Markdown(open("/kaggle/working/outputs/results/summary.md").read()))
display(Image("/kaggle/working/outputs/results/figures/robustness_curves.png"))

## Phase 5b — Harder stress test
The default factors (0.5–1.5x) barely moved any method. `configs/stress.yaml` widens them to
0.25–4x and adds two perturbations: **warp** (first half of the sign at s x, second half at 1/s x)
and **hold** (pause at the sign's peak; slow-down only). Same folds, seeds and methods; results go
to `outputs/results_stress/`. Resumable.

In [ ]:
!python -m tempobis train --config $SCFG 2>&1 | tail -80

In [ ]:
display(Markdown(open("/kaggle/working/outputs/results_stress/summary.md").read()))
display(Image("/kaggle/working/outputs/results_stress/figures/robustness_curves.png"))

## Phase 6 — Error analysis

In [ ]:
!python -m tempobis analyze --config $CFG

In [ ]:
display(Image("/kaggle/working/outputs/results/figures/confusion_uniform.png"))
display(Image("/kaggle/working/outputs/results/figures/confusion_motion_aware.png"))

## Save outputs
Download `tempobis_results.zip` (Output tab) and send `summary.md`, `error_analysis_*.md`,
`audit_summary.json` and the figures back for the write-up. Models for the demo are in `results/models/`.

In [ ]:
!cd /kaggle/working/outputs && zip -qr /kaggle/working/tempobis_results.zip results audit landmark_stats.csv -x "results/models/*"
!cd /kaggle/working/outputs && zip -qr /kaggle/working/tempobis_models.zip results/models results/config_used.json
!ls -lh /kaggle/working/*.zip